In [1]:
import pandas as pd
import numpy as np
from sklearn.neighbors import KDTree
import matplotlib.pyplot as plt
import seaborn as sns
import hnswlib
from config import tcr_embeddings_path, mhc_embeddings_path, peptide_embeddings_path, all_relations_path, CMV_dataset_path, train_path, val_path, test_path
import pickle
import torch
import torch.nn.functional as F
import torch
from threading import Lock
from concurrent.futures import ThreadPoolExecutor
from torch.utils.data import Dataset
from sklearn.neighbors import NearestNeighbors
import os
import dhg
from typing import List, Tuple, Optional, Union
from itertools import chain

/home/tarnickil/.conda/envs/mgr_thesis/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
with open(tcr_embeddings_path, "rb") as f:
    tcr_embbedings = pickle.load(f)

with open(mhc_embeddings_path, "rb") as f:
    mhc_embbedings = pickle.load(f)

with open(peptide_embeddings_path, "rb") as f:
    peptide_embbedings = pickle.load(f)
    
with open(all_relations_path, "rb") as f:
    all_relations = pickle.load(f)

with open(CMV_dataset_path, "rb") as f:
    cmv_dataset = pickle.load(f)

In [3]:
with open(train_path, "rb") as f:
    train_data = pickle.load(f)
    
with open(val_path, "rb") as f:
    val_data = pickle.load(f)

with open(test_path, "rb") as f:
    test_data = pickle.load(f)

In [4]:
map_dict = {"Non-binding":0, "Binding":1}
all_relations["Binding"] = all_relations["Binding"].replace(map_dict)
tcr_embbedings.rename({"Name":"TCR_name"}, axis=1, inplace=True)

/tmp/ipykernel_137009/292328124.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  all_relations["Binding"] = all_relations["Binding"].replace(map_dict)


In [17]:
cmv_dataset[cmv_dataset["HLA_MHC"] == "HLA-A2402"]["Binding"].value_counts()

Binding
0    66927
1       30
Name: count, dtype: int64

In [ ]:
t = cmv_dataset[cmv_dataset["Binding"]==1][:100].index.to_numpy()
# all_relations[all_relations["TCR_name"] == "TCR_43"]

In [ ]:
alpha_pd = pd.DataFrame(np.stack(tcr_embbedings[["Embeddings_alpha"]].values.flatten()))
alpha_pd["index"] = alpha_pd.index
beta_pd = pd.DataFrame(np.stack(tcr_embbedings[["Embeddings_beta"]].values.flatten()))
beta_pd["index"] = beta_pd.index
alpha_beta_pd=pd.merge(
    left=alpha_pd,
    right=beta_pd,
    how="left",
    on="index"
)
alpha_beta_pd["TCR_name"] = tcr_embbedings["TCR_name"].values
alpha_beta_pd.drop("index", axis=1, inplace=True)
# alpha_beta_pd.loc[[1,2,4,5,8,9], alpha_beta_pd.columns != 'TCR_name']

In [ ]:
# alpha_beta_pd # 66957 66956
# tcr_embbedings #66957 66956
alpha_beta_pd

,0_x,1_x,2_x,3_x,4_x,5_x,6_x,7_x,8_x,9_x,...,631_y,632_y,633_y,634_y,635_y,636_y,637_y,638_y,639_y,TCR_name
0,-0.178072,0.047997,0.141127,-0.009966,-0.161461,-0.086877,-0.011707,-0.281793,-0.117906,0.150621,...,0.237778,-0.478275,0.155913,0.106157,0.149361,-0.095889,0.126096,-0.273895,-0.139861,TCR_1
1,-0.155885,0.045211,0.186540,-0.000201,-0.136094,-0.101367,-0.024795,-0.290179,-0.115840,0.193600,...,0.152638,-0.630272,0.142683,0.021747,0.197445,-0.077530,0.171386,-0.265552,-0.083890,TCR_2
2,-0.184542,0.004414,0.235234,-0.032479,-0.188835,-0.056024,0.061203,-0.262615,-0.078777,0.237720,...,0.255104,-0.558620,0.207630,0.028521,0.208704,-0.105614,0.196280,-0.186920,-0.113264,TCR_3
3,-0.135676,0.069660,0.176683,-0.060076,-0.134584,-0.051249,0.019765,-0.241480,-0.024446,0.177347,...,0.132520,-0.582464,0.238804,-0.045056,0.173232,-0.053518,0.110144,-0.249446,-0.115899,TCR_4
4,-0.172737,0.025500,0.235653,-0.058014,-0.218861,-0.047917,0.000261,-0.188171,-0.078360,0.221932,...,0.169897,-0.715993,0.238676,-0.008413,0.138062,-0.050015,0.148664,-0.211657,-0.108754,TCR_5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66952,-0.194515,-0.006650,0.137349,-0.131500,-0.306162,-0.042293,-0.026948,-0.077727,-0.048534,0.108062,...,0.157705,-0.666635,0.258532,-0.044375,0.167840,0.041720,0.182558,-0.275042,-0.125996,TCR_66953
66953,-0.238259,0.101412,0.235044,-0.066151,-0.246057,0.043120,0.045727,-0.286881,-0.051623,0.148888,...,0.260768,-0.461189,0.135645,0.087055,0.184785,-0.017544,0.181941,-0.264334,-0.078694,TCR_66954
66954,-0.216587,0.088009,0.239563,-0.080803,-0.124314,-0.126172,-0.020690,-0.306471,-0.086623,0.149490,...,0.230898,-0.573675,0.275421,-0.007072,0.285312,-0.004438,0.232176,-0.213936,-0.056068,TCR_66955
66955,-0.207422,0.006047,0.171977,-0.143296,-0.164765,-0.112295,-0.056780,-0.202620,-0.155826,0.145379,...,0.117178,-0.406453,0.167587,-0.008485,0.234288,-0.028525,0.023194,-0.273080,-0.127592,TCR_66956


In [ ]:
Hypergraf_df = all_relations[["Name", "HLA_MHC", "Epitop", "TCR_name"]]
index = Hypergraf_df.index.values
rng = np.random.default_rng()
r_i = rng.choice(index, size=10)
test_df = Hypergraf_df.iloc[r_i,:]
test_df
for i in test_df.columns[1:-1]:
    print(test_df[i].value_counts())

HLA_MHC
HLA-A2402    4
HLA-A0201    3
HLA-B0702    2
HLA-B0801    1
Name: count, dtype: int64
Epitop
pp65_CMV_binder               2
WT1-(235-243)236M_Y_binder    2
Gag-protein_HIV_binder        2
EBNA-6_EBV_binder             1
NC_binder                     1
BZLF1_EBV_binder              1
PSA146-154_binder             1
Name: count, dtype: int64


In [ ]:
# def graph_knn(data:pd.DataFrame):
#     '''
#     data: Zbiór danych zawirający embeddingi ciągów alpha oraz beta 
#     '''
#     embeddings = data.iloc[:, :-1].values  # shape: (n_punktów, n_wymiarów)
#     n, dim = embeddings.shape

#     # Normalizacja (ważne przy cosine!)
#     norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
#     embeddings_norm = embeddings / norms

#     # --- Budowanie indeksu ---
#     index = hnswlib.Index(space='cosine', dim=dim)
#     #                     ↑
#     #              metryka podobieństwa:
#     #              'cosine' — dla embeddingów językowych/biologicznych
#     #              'l2'     — odległość euklidesowa
#     #              'ip'     — iloczyn skalarny (dot product)

#     index.init_index(
#         max_elements=n,       # ile punktów maksymalnie
#         ef_construction=200,  # dokładność budowania (wyżej = lepiej, wolniej)
#         M=16,
#         random_seed = 100             # liczba krawędzi na węzeł (wyżej = lepiej, więcej RAM)
#     )

#     # Dodaj punkty Z ich oryginalnymi indeksami z DataFrame
#     index.add_items(embeddings_norm, ids=np.arange(n))

#     return index, embeddings_norm

def graph_knn(data: pd.DataFrame):
    '''
    data: Zbiór danych zawierający embeddingi ciągów alpha oraz beta
    Zwraca: model sklearn (NearestNeighbors) oraz znormalizowane embeddingi (ndarray)
    '''
    embeddings = data.iloc[:, :-1].values  # (n_punktów, n_wymiarów)
    n, dim = embeddings.shape

    # Normalizacja L2 (ważne przy cosine)
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    embeddings_norm = embeddings / norms

    # Deterministyczny indeks: brute-force, metryka cosine
    nn_model = NearestNeighbors(
        n_neighbors=200,          # domyślne k, można zmienić przy zapytaniu
        metric='cosine',
        algorithm='brute'
    )
    nn_model.fit(embeddings_norm)

    return nn_model, embeddings_norm

In [ ]:
knn_graph, norm_embeddings = graph_knn(alpha_beta_pd)

In [ ]:
# all_relations[all_relations["Binding"]==1] #TCR_108
# all_relations[all_relations["TCR_name"]=="TCR_108"]
# tcr_embbedings
# all_relations

In [ ]:
def graph_edge_chooser(data_all, target_edges, knn_data, X_TCR, X_MHC, k_sim=10):
    '''
    data_all -> pd.Dataframe Zawierający wszystkie informacje o relacjach
    target_edges -> np.array Zawierająćy wszystkie indeksy relacji dla który powstanie sub-graph
    knn_data -> pd.DataFrame Zawierający przygotowane embeddingi CDR3 do budowania grafu knn
    X_TCR, X_MHC -> pd.DataFrame Zawierający embeddingi dla każdego z typów wierzchołków
    k_sim -> Liczba sąsiadów do utworzenia krawędzi similarity wewnątrz V_TCR_final
    '''

    knn_graph, norm_embeddings = graph_knn(knn_data)
    
    results = []

    for edge in target_edges:
        target_relation = data_all.iloc[edge,:][["TCR_name", "HLA_MHC", "Name", "Binding"]].values
        embedding_idx = knn_data[knn_data["TCR_name"] == target_relation[0]].index
        target_tcr_embedding = norm_embeddings[embedding_idx]
        neighbours, cos_dist = knn_graph.knn_query(target_tcr_embedding[:,:-1], k=200)
        cos_dist = np.round(cos_dist[0].astype(np.float32),4)
        neighbours_names = X_TCR.iloc[neighbours.flatten(),:]["TCR_name"].values
        cos_df = pd.DataFrame({"TCR_name":neighbours_names, "cos_sim":cos_dist})
        neigbours_relations = data_all[data_all["TCR_name"].isin(neighbours_names)][["TCR_name", "HLA_MHC", "Name", "Binding"]]

        neigbours_relations["czy_target"] = np.where(neigbours_relations.index.isin([edge]), 1, 0)
        neigbours_relations = pd.merge(
            left=neigbours_relations,
            right=cos_df,
            on="TCR_name",
            how="left"
        )

        neighbours_tier1 = neigbours_relations[(neigbours_relations["HLA_MHC"] == target_relation[1])]
        
        V_TCR_final = neighbours_tier1.reset_index(drop=True)

        # =====================================================================
        # BUDOWA STRUKTURY GRAFU (H, wagi krawędzi, target proximity flag)
        # =====================================================================
        N = len(V_TCR_final)
        # target_idx = int(V_TCR_final.index[V_TCR_final["czy_target"] == 1][0])
        target_idx = V_TCR_final[V_TCR_final["czy_target"]==1].index.to_numpy()[0]

        local_emb_idx = [
            knn_data[knn_data["TCR_name"] == name].index[0]
            for name in V_TCR_final["TCR_name"]
        ]
        local_embeddings = torch.tensor(
            norm_embeddings[local_emb_idx, :-1], dtype=torch.float32
        )
        print(f"[A] local_emb_idx[:10]: {[int(x) for x in local_emb_idx[:10]]}")
        print(f"[A] local_embeddings.sum(): {float(local_embeddings.sum()):.10f}")

        sim = local_embeddings @ local_embeddings.T
        sim.fill_diagonal_(-float("inf"))

        topk_vals, topk_idx = sim.topk(k=min(k_sim, N - 1), dim=1)

        edges = {}
        for i in range(N):
            for k in range(topk_idx.shape[1]):
                j = int(topk_idx[i, k])
                key = (min(i, j), max(i, j))
                w = float(topk_vals[i, k])
                edges[key] = max(edges.get(key, w), w)

        edge_list = list(edges.keys())
        print(f"[B] M={len(edge_list)}  first 5: {edge_list[:5]}  last 5: {edge_list[-5:]}")
        M = len(edge_list)

        H = torch.zeros(N, M)
        for m, (i, j) in enumerate(edge_list):
            H[i, m] = 1
            H[j, m] = 1
        e_weight = torch.tensor([edges[e] for e in edge_list], dtype=torch.float32)

        A = (H @ H.T > 0).float()
        A.fill_diagonal_(0)
        one_hop = A[target_idx].bool()
        two_hop_raw = (A @ A)[target_idx].bool()
        two_hop = two_hop_raw & ~one_hop
        two_hop[target_idx] = False

        target_flag = torch.zeros(N, 4)
        target_flag[target_idx, 0] = 1
        target_flag[one_hop, 1] = 1
        target_flag[two_hop, 2] = 1
        target_flag[(target_flag.sum(1) == 0), 3] = 1

        # 1. Embeddingi ESM2(α+β) per węzeł, w tej samej kolejności co V_TCR_final
        # X_TCR ma kolumnę TCR_name + kolumny embeddingów (np. emb_0, ..., emb_d)
        X_TCR_indexed = X_TCR.set_index("TCR_name")
        selected = X_TCR_indexed.loc[V_TCR_final["TCR_name"].values]
        alpha_emb = np.stack(selected["Embeddings_alpha"].values)  # [N, d_alpha]
        beta_emb = np.stack(selected["Embeddings_beta"].values)    # [N, d_beta]
        X_esm = torch.tensor(
            np.concatenate([alpha_emb, beta_emb], axis=1),
            dtype=torch.float32,
        )  # [N, d_alpha + d_beta]
        
        # 2. One-hot MHC* dla wszystkich węzłów (cały podgraf w kontekście jednego MHC)
        all_mhc = sorted(data_all["HLA_MHC"].unique())
        mhc_to_id = {m: i for i, m in enumerate(all_mhc)}
        n_mhc = len(all_mhc)
        mhc_id = mhc_to_id[target_relation[1]]
        X_mhc = torch.zeros(N, n_mhc)
        X_mhc[:, mhc_id] = 1.0

        # 3. Konkatenacja: [ESM2 | one-hot MHC | target flag]
        X = torch.cat([X_esm, X_mhc, target_flag], dim=1)  # [N, d_esm + n_mhc + 4]

        results.append({
            "V_TCR_final": V_TCR_final,
            "H": H,
            "edge_list": edge_list,
            "e_weight": e_weight,
            "target_flag": target_flag,
            "X": X,                              # dodane
            "target_relation": target_relation,
        })

    return results

In [ ]:
r = graph_edge_chooser(cmv_dataset, target_edges=[t[0]], knn_data=alpha_beta_pd, X_TCR = tcr_embbedings, X_MHC = mhc_embbedings)

AttributeError: 'NearestNeighbors' object has no attribute 'knn_query'

In [ ]:
train_positive = train_data[train_data["Binding"]==1]
train_negative = train_data[train_data["Binding"]==0]
train_positive_count = train_positive.shape[0]
train_negative = train_negative.sample(n=2*train_positive_count, replace=False, random_state=42)
training_data = pd.concat([train_positive, train_negative], axis=0)
training_indexs = cmv_dataset[cmv_dataset["Name"].isin(training_data["Name"].values)].index.to_numpy()
training_indexs

array([     6,      9,     17, ..., 133868, 133886, 133897],
      shape=(26418,))

In [ ]:
# training_graphs_struct = graph_edge_chooser(cmv_dataset, target_edges=training_indexs, knn_data=alpha_beta_pd, X_TCR = tcr_embbedings, X_MHC = mhc_embbedings)

In [ ]:
val_positive = val_data[val_data["Binding"]==1]
val_negative = val_data[val_data["Binding"]==0]
val_positive_count = val_positive.shape[0]
val_positive_count
val_negative = val_negative.sample(n=2*val_positive_count, replace=False, random_state=42)
validation_data = pd.concat([val_positive, val_negative], axis=0)
validation_indexs = cmv_dataset[cmv_dataset["Name"].isin(validation_data["Name"].values)].index.to_numpy()
validation_indexs

array([     1,     15,     18, ..., 133885, 133905, 133910], shape=(7593,))

In [ ]:
# validation_graphs_struct = graph_edge_chooser(cmv_dataset, target_edges=training_indexs, knn_data=alpha_beta_pd, X_TCR = tcr_embbedings, X_MHC = mhc_embbedings)

In [ ]:
test_positive = test_data[test_data["Binding"]==1]
test_negative = test_data[test_data["Binding"]==0]
test_positive_count = test_positive.shape[0]
test_positive_count
test_negative = test_negative.sample(n=2*test_positive_count, replace=False, random_state=42)
testing_data = pd.concat([test_positive, test_negative], axis=0)
test_indexs = cmv_dataset[cmv_dataset["Name"].isin(testing_data["Name"].values)].index.to_numpy()
test_indexs

array([     0,     34,     55, ..., 133855, 133861, 133901], shape=(7707,))

In [ ]:
# test_graphs_struct = graph_edge_chooser(cmv_dataset, target_edges=training_indexs, knn_data=alpha_beta_pd, X_TCR = tcr_embbedings, X_MHC = mhc_embbedings)
# # 36955

In [ ]:
class _SklearnKnnWrapper:
    """Adapter sklearn -> hnswlib-style API: knn_query(q, k) -> (indices, dist)."""
 
    def __init__(self, model):
        self._model = model
 
    def knn_query(self, query, k):
        distances, indices = self._model.kneighbors(query, n_neighbors=k)
        return indices, distances
 
 
def graph_knn_cl(data: pd.DataFrame, n_jobs=-1):
    """
    Deterministyczny zamiennik hnswlib-owego graph_knn.
 
    n_jobs : rdzenie dla sklearn.kneighbors. Ustaw 1 gdy używasz outer
             ThreadPoolExecutor (uniknięcie zagnieżdżania).
 
    Zwraca: (knn_graph, embeddings_norm) — sygnatura jak oryginał.
    """
    embeddings = data.iloc[:, :-1].values  # bez kolumny TCR_name
    n, dim = embeddings.shape
 
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    embeddings_norm = embeddings / norms
 
    nn = NearestNeighbors(
        n_neighbors=200,
        metric='cosine',
        algorithm='brute',
        n_jobs=n_jobs,
    )
    nn.fit(embeddings_norm)
 
    return _SklearnKnnWrapper(nn), embeddings_norm
 
 
# =============================================================================
# GŁÓWNA FUNKCJA
# =============================================================================
def graph_edge_chooser_cl(
    data_all,
    target_edges,
    knn_graph,
    norm_embeddings,
    knn_data,
    X_TCR,
    X_MHC,
    output_dir=None,
    k_sim=10,
    n_jobs=8,
    overwrite=False,
    quiet=False,
):
    """
    output_dir : 
        - None (lub "" lub False): TRYB IN-MEMORY. Funkcja zwraca listę dictów
          identyczną z oryginalną wersją {V_TCR_final, H, edge_list, e_weight,
          target_flag, X, target_relation}. Do szybkiego testowania.
        - "ścieżka/do/folderu": TRYB DISK. Każdy edge zapisywany do
          edge_{idx}.pt; funkcja zwraca listę ścieżek.
 
    Pozostałe argumenty bez zmian.
    """
    save_to_disk = bool(output_dir)
 
    if save_to_disk:
        os.makedirs(output_dir, exist_ok=True)
 
    # =========================================================================
    # PRECOMPUTE
    # =========================================================================
    all_mhc = sorted(data_all["HLA_MHC"].unique())
    mhc_to_id = {m: i for i, m in enumerate(all_mhc)}
    n_mhc = len(all_mhc)
    X_TCR_indexed = X_TCR.set_index("TCR_name")
 
    tcr_to_emb_idx = dict(
        zip(knn_data["TCR_name"].values, knn_data.index.values)
    )
    known_tcrs_knn = set(tcr_to_emb_idx.keys())
 
    alpha_mat = np.stack(X_TCR_indexed["Embeddings_alpha"].values)
    beta_mat = np.stack(X_TCR_indexed["Embeddings_beta"].values)
    esm_mat = np.concatenate([alpha_mat, beta_mat], axis=1)
    xtcr_name_to_row = {n: i for i, n in enumerate(X_TCR_indexed.index)}
    known_tcrs_xtcr = set(xtcr_name_to_row.keys())
 
    print_lock = Lock()
 
    def log(msg):
        if not quiet:
            with print_lock:
                print(msg)
 
    # =========================================================================
    # JEDEN EDGE
    # Zwraca: (result, skip_info)
    #   - w trybie disk: result = ścieżka albo None
    #   - w trybie memory: result = dict albo None
    # =========================================================================
    def process_edge(edge):
        if save_to_disk:
            out_path = os.path.join(output_dir, f"edge_{edge}.pt")
            if not overwrite and os.path.exists(out_path):
                return out_path, None
 
        # 1. TARGET
        target_relation = data_all.iloc[edge, :]
        target_tcr_name = target_relation["TCR_name"]
        target_hla = target_relation["HLA_MHC"]
 
        # 2. EMBEDDING TARGETU
        if target_tcr_name not in tcr_to_emb_idx:
            log(f"POMINIĘTO {edge}: brak embeddingu dla {target_tcr_name}")
            return None, (edge, "no_embedding")
 
        target_emb_idx = int(tcr_to_emb_idx[target_tcr_name])
        target_tcr_embedding = norm_embeddings[target_emb_idx:target_emb_idx + 1]
 
        # 3. KNN — pełny 1280-D query (różnica vs oryginał: bez [:, :-1])
        neighbours, cos_dist = knn_graph.knn_query(target_tcr_embedding, k=200)
        cos_dist = np.round(cos_dist[0].astype(np.float32), 4)
        neighbours_names = X_TCR.iloc[neighbours.flatten(), :]["TCR_name"].values
 
        cos_df = (
            pd.DataFrame({"TCR_name": neighbours_names, "cos_sim": cos_dist})
            .drop_duplicates(subset="TCR_name", keep="first")
        )
 
        neigbours_relations = (
            data_all.loc[
                data_all["TCR_name"].isin(neighbours_names),
                ["TCR_name", "HLA_MHC", "Name", "Binding"]
            ]
            .merge(cos_df, on="TCR_name", how="left")
        )
 
        neighbours_tier1 = neigbours_relations[
            neigbours_relations["HLA_MHC"] == target_hla
        ].copy()
 
        # 4. WYMUSZENIE TARGETU
        mask_dup_target = (
            (neighbours_tier1["TCR_name"] == target_tcr_name)
            & (neighbours_tier1["HLA_MHC"] == target_hla)
        )
        neighbours_tier1 = neighbours_tier1.loc[~mask_dup_target].copy()
        neighbours_tier1["czy_target"] = 0
 
        target_to_add = pd.DataFrame([{
            "TCR_name":   target_tcr_name,
            "HLA_MHC":    target_hla,
            "Name":       target_relation["Name"],
            "Binding":    target_relation["Binding"],
            "cos_sim":    0.0,
            "czy_target": 1,
        }])[neighbours_tier1.columns]
 
        V_TCR_final = pd.concat(
            [neighbours_tier1, target_to_add], axis=0, ignore_index=True
        )
 
        n_targets = int((V_TCR_final["czy_target"] == 1).sum())
        if n_targets != 1:
            log(f"POMINIĘTO {edge}: target_count={n_targets}")
            return None, (edge, f"target_count={n_targets}")
 
        # 5. SKIP CHECKS
        names_in_subgraph = V_TCR_final["TCR_name"].tolist()
 
        missing_in_knn = [n for n in names_in_subgraph if n not in known_tcrs_knn]
        if missing_in_knn:
            log(f"POMINIĘTO {edge}: {len(missing_in_knn)} bez embeddingu")
            return None, (edge, f"missing_in_knn={len(missing_in_knn)}")
 
        missing_in_xtcr = [n for n in names_in_subgraph if n not in known_tcrs_xtcr]
        if missing_in_xtcr:
            log(f"POMINIĘTO {edge}: {len(missing_in_xtcr)} bez X_TCR")
            return None, (edge, f"missing_in_xtcr={len(missing_in_xtcr)}")
 
        if target_hla not in mhc_to_id:
            log(f"POMINIĘTO {edge}: HLA_MHC={target_hla} unknown")
            return None, (edge, "unknown_mhc")
 
        # =====================================================================
        # BUDOWA STRUKTURY GRAFU
        # =====================================================================
        N = len(V_TCR_final)
        target_idx = int(V_TCR_final.index[V_TCR_final["czy_target"] == 1][0])
 
        local_emb_idx = [tcr_to_emb_idx[name] for name in V_TCR_final["TCR_name"]]
        local_embeddings = torch.tensor(
            norm_embeddings[local_emb_idx, :-1], dtype=torch.float32
        )
 
        sim = local_embeddings @ local_embeddings.T
        sim.fill_diagonal_(-float("inf"))
        topk_vals, topk_idx = sim.topk(k=min(k_sim, N - 1), dim=1)
 
        edges = {}
        for i in range(N):
            for k in range(topk_idx.shape[1]):
                j = int(topk_idx[i, k])
                key = (min(i, j), max(i, j))
                w = float(topk_vals[i, k])
                edges[key] = max(edges.get(key, w), w)
 
        edge_list = list(edges.keys())
        M = len(edge_list)
 
        if M > 0:
            edge_arr = torch.tensor(edge_list, dtype=torch.long)
            col_idx = torch.arange(M)
            H = torch.zeros(N, M)
            H[edge_arr[:, 0], col_idx] = 1
            H[edge_arr[:, 1], col_idx] = 1
        else:
            H = torch.zeros(N, 0)
 
        e_weight = torch.tensor([edges[e] for e in edge_list], dtype=torch.float32)
 
        A = (H @ H.T > 0).float()
        A.fill_diagonal_(0)
        one_hop = A[target_idx].bool()
        two_hop_raw = (A @ A)[target_idx].bool()
        two_hop = two_hop_raw & ~one_hop
        two_hop[target_idx] = False
 
        target_flag = torch.zeros(N, 4)
        target_flag[target_idx, 0] = 1
        target_flag[one_hop, 1] = 1
        target_flag[two_hop, 2] = 1
        target_flag[(target_flag.sum(1) == 0), 3] = 1
 
        rows = [xtcr_name_to_row[n] for n in V_TCR_final["TCR_name"]]
        X_esm = torch.tensor(esm_mat[rows], dtype=torch.float32)
 
        mhc_id = mhc_to_id[target_hla]
        X_mhc = torch.zeros(N, n_mhc)
        X_mhc[:, mhc_id] = 1.0
 
        X = torch.cat([X_esm, X_mhc, target_flag], dim=1)
 
        # =====================================================================
        # WYNIK — zależny od trybu
        # =====================================================================
        if save_to_disk:
            sample = {
                "V_TCR_final":     V_TCR_final,
                "H":               H,
                "edge_list":       edge_list,
                "e_weight":        e_weight,
                "target_flag":     target_flag,
                "X":               X,
                "target_relation": target_relation,
                "edge_idx":        edge,
            }
            tmp_path = out_path + ".tmp"
            torch.save(sample, tmp_path)
            os.replace(tmp_path, out_path)
            return out_path, None
        else:
            # Format 1:1 z oryginalną funkcją (bez edge_idx)
            result = {
                "V_TCR_final":     V_TCR_final,
                "H":               H,
                "edge_list":       edge_list,
                "e_weight":        e_weight,
                "target_flag":     target_flag,
                "X":               X,
                "target_relation": target_relation,
            }
            return result, None
 
    # =========================================================================
    # WYKONANIE
    # =========================================================================
    results = []
    skipped = []
 
    if n_jobs == 1:
        for edge in target_edges:
            res, skip = process_edge(edge)
            if res is not None:
                results.append(res)
            if skip is not None:
                skipped.append(skip)
    else:
        with ThreadPoolExecutor(max_workers=n_jobs) as ex:
            for res, skip in ex.map(process_edge, target_edges):
                if res is not None:
                    results.append(res)
                if skip is not None:
                    skipped.append(skip)
 
    # Manifest — tylko w trybie disk
    if save_to_disk:
        manifest_path = os.path.join(output_dir, "manifest.txt")
        with open(manifest_path, "w") as f:
            for p in sorted(results):
                f.write(p + "\n")
        print(f"\nZapisano {len(results)} sub-grafów do {output_dir}")
    else:
        print(f"\nZbudowano {len(results)} sub-grafów w pamięci")
 
    if skipped:
        print(f"Pominięto {len(skipped)} edge'y:")
        for e, reason in skipped[:20]:
            print(f"  edge={e}: {reason}")
        if len(skipped) > 20:
            print(f"  ... i {len(skipped) - 20} więcej")
 
    return results
 
 
# =============================================================================
# DATASET DLA PYTORCH DATALOADER
# =============================================================================
class TCRSubgraphDataset(Dataset):
    """
    Dataset zwracający sub-grafy zapisane przez graph_edge_chooser (tryb disk).
 
    Konstruktor przyjmuje:
      - ścieżkę do katalogu (znajdzie manifest.txt lub przeskanuje edge_*.pt)
      - ścieżkę do manifest.txt
      - listę ścieżek
 
    UWAGA: subgraphy mają RÓŻNE N i M — użyj `collate_fn=tcr_subgraph_collate`.
    """
 
    def __init__(self, source):
        if isinstance(source, (list, tuple)):
            self.paths = list(source)
        elif os.path.isdir(source):
            manifest = os.path.join(source, "manifest.txt")
            if os.path.exists(manifest):
                with open(manifest) as f:
                    self.paths = [line.strip() for line in f if line.strip()]
            else:
                self.paths = sorted(
                    os.path.join(source, f)
                    for f in os.listdir(source)
                    if f.startswith("edge_") and f.endswith(".pt")
                )
        elif os.path.isfile(source):
            with open(source) as f:
                self.paths = [line.strip() for line in f if line.strip()]
        else:
            raise ValueError(f"Nie potrafię zinterpretować source: {source}")
 
        if len(self.paths) == 0:
            raise RuntimeError(f"Pusty Dataset — brak plików w {source}")
 
    def __len__(self):
        return len(self.paths)
 
    def __getitem__(self, idx):
        return torch.load(self.paths[idx], weights_only=False)
 
 
def tcr_subgraph_collate(batch):
    """Trywialny collate: zwraca listę próbek (sub-grafy mają różne wymiary)."""
    return batch

In [ ]:
knn_graph, norm_embeddings = graph_knn_cl(alpha_beta_pd)

r_1 = graph_edge_chooser_cl(
    cmv_dataset,
    target_edges=[t[0]],
    knn_graph=knn_graph,
    norm_embeddings=norm_embeddings,
    knn_data=alpha_beta_pd,
    X_TCR=tcr_embbedings,
    X_MHC=mhc_embbedings,
    # output_dir nie podany -> in-memory
    n_jobs=1,
)   #knn_graph,norm_embeddings,

r_2 = graph_edge_chooser_cl(
    cmv_dataset,
    target_edges=[t[0]],
    knn_graph=knn_graph,
    norm_embeddings=norm_embeddings,
    knn_data=alpha_beta_pd,
    X_TCR=tcr_embbedings,
    X_MHC=mhc_embbedings,
    # output_dir nie podany -> in-memory
    n_jobs=1,
)   #knn_graph,norm_embeddings,


Zbudowano 1 sub-grafów w pamięci

Zbudowano 1 sub-grafów w pamięci


In [ ]:
r_2[0]["target_relation"]

Name        A0301_KLGGALQAK_IE-1_CMV_binder_TCR_10010
pMHC                  A0301_KLGGALQAK_IE-1_CMV_binder
CDR3a                                  CALTSNTNAGKSTF
CDR3b                              CASSLRDRLITGANVLTF
Epitop                                IE-1_CMV_binder
TCR_name                                    TCR_10010
HLA_MHC                                     HLA-A0301
Binding                                             1
Count                                               2
Name: 15, dtype: object

In [ ]:
# np.sort(r_o[0]["V_TCR_final"]["TCR_name"].values) == np.sort(r_p[0]["V_TCR_final"]["TCR_name"].values)
r_2[0]["V_TCR_final"].equals(r_1[0]["V_TCR_final"])

True

In [ ]:
print("V_TCR_final", r_o[0]["V_TCR_final"].shape)
print("H", r_o[0]["H"].shape)
print("edge_list", len(r_o[0]["edge_list"]))
print("e_weight", r_o[0]["e_weight"].shape)
print("target_flag", r_o[0]["target_flag"].shape)
print("X", r_o[0]["X"].shape)
print("target_relation", r_o[0]["target_relation"].shape)

V_TCR_final (200, 6)
H torch.Size([200, 1423])
edge_list 1423
e_weight torch.Size([1423])
target_flag torch.Size([200, 4])
X torch.Size([200, 1286])
target_relation (4,)


In [ ]:
print("V_TCR_final", r_p[0]["V_TCR_final"].shape)
print("H", r_p[0]["H"].shape)
print("edge_list", len(r_p[0]["edge_list"]))
print("e_weight", r_p[0]["e_weight"].shape)
print("target_flag", r_p[0]["target_flag"].shape)
print("X", r_p[0]["X"].shape)
print("target_relation", r_p[0]["target_relation"].shape)

V_TCR_final (200, 6)
H torch.Size([200, 1426])
edge_list 1426
e_weight torch.Size([1426])
target_flag torch.Size([200, 4])
X torch.Size([200, 1286])
target_relation (4,)
